# SHIPP Stage 6 — Silver candidate pairs

`silver_listings` + `silver_requests` → `silver_candidate_pairs`
**Grain:** one row = one (listing, request) pair that passes every hard eligibility rule.
Logic: `data_pipeline/silver/pairs.py` (unit-tested). Settings: `config/settings.py`.

| Rule | Setting |
|---|---|
| Listing status | `ELIGIBLE_LISTING_STATUSES`; coordinates present; `available_until` in the future |
| Request status | `ELIGIBLE_REQUEST_STATUSES`; coordinates present; `need_by_date` in the future |
| Category | equal (both normalised in Silver) |
| Timing | listing available on/before `need_by_date` |
| Not self | `donor_id != requester_id` |
| Pre-filter | straight-line ≤ `MAX_STRAIGHT_LINE_KM` |

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_candidate_pairs`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
from data_pipeline.silver.pairs import PAIR_COLUMNS, build_candidate_pairs

SILVER_CANDIDATE_PAIRS = f"{CATALOG}.{SILVER_SCHEMA}.silver_candidate_pairs"
TARGET_TABLE = SILVER_CANDIDATE_PAIRS
EXPECTED_COLUMNS = PAIR_COLUMNS + ["silver_processed_at"]

pairs_df = (
    build_candidate_pairs(spark.table(SILVER_LISTINGS), spark.table(SILVER_REQUESTS), now_ntz())
    .withColumn("silver_processed_at", F.current_timestamp())
)
pair_count = pairs_df.count()
print("Candidate pairs:", pair_count)
display(pairs_df.select("listing_id", "request_id", "category", "straight_line_km").orderBy("request_id", "straight_line_km"))

In [0]:
run_quality_gate({
    "duplicate pair_id": duplicate_key_count(pairs_df, "pair_id"),
    "duplicate (listing, request)": duplicate_key_count(pairs_df, "listing_id", "request_id"),
    "self pair": pairs_df.filter(F.col("donor_id") == F.col("requester_id")).count(),
    "null route key": pairs_df.filter(F.col("origin_key").isNull() | F.col("dest_key").isNull()).count(),
}, row_count=pair_count, allow_empty=True)
if pair_count == 0:
    print("WARNING — zero pairs. Run validation/03_demo_data_readiness to see why.")

written = write_overwrite(pairs_df, TARGET_TABLE)
assert written == pair_count, f"WRITE VALIDATION FAILED: wrote {written}, expected {pair_count}"
assert_exact_schema(TARGET_TABLE, EXPECTED_COLUMNS)

summary = {"table": TARGET_TABLE, "rows": written, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))

In [0]:
run_quality_gate(
    {
        "duplicate pair_id": duplicate_key_count(
            pairs_df,
            "pair_id",
        ),

        "duplicate (listing, request)": duplicate_key_count(
            pairs_df,
            "listing_id",
            "request_id",
        ),

        "self pair": pairs_df.filter(
            F.col("donor_id")
            == F.col("requester_id")
        ).count(),

        "null route key": pairs_df.filter(
            F.col("origin_key").isNull()
            | F.col("dest_key").isNull()
        ).count(),

        # Final hardening
        "negative straight-line distance": pairs_df.filter(
            F.col("straight_line_km") < 0
        ).count(),

        "pair beyond prefilter limit": pairs_df.filter(
            F.col("straight_line_km")
            > MAX_STRAIGHT_LINE_KM
        ).count(),
    },
    row_count=pair_count,
    allow_empty=True,
)

if pair_count == 0:
    print(
        "WARNING — zero eligible pairs. "
        "Check current Silver marketplace state."
    )

written = write_overwrite(
    pairs_df,
    TARGET_TABLE,
)

assert written == pair_count, (
    f"WRITE VALIDATION FAILED: "
    f"wrote {written}, expected {pair_count}"
)

assert_exact_schema(
    TARGET_TABLE,
    EXPECTED_COLUMNS,
)

summary = {
    "table": TARGET_TABLE,
    "rows": written,
    "status": "PASS",
}

print(summary)

dbutils.notebook.exit(
    json.dumps(summary)
)